# Day 12: Contrastive Learning for Activity Cliff Prediction

**Goal:** Finetune a D-MPNN (Chemprop) using contrastive learning on analog pairs to better distinguish activity cliffs

## Two Approaches:

### A. Full-Molecule Contrastive Learning
- Find analog pairs with high Tanimoto similarity but different pEC50 values
- Train model to push similar molecules with different activities apart in embedding space
- Use triplet loss or NT-Xent (SimCLR) loss

### B. Pocket-Masking Contrastive Learning (PXR-Specific)
- Identify atoms near F288/W299/Y306 hydrophobic π-trap using:
  - Docking results (if available)
  - OR heuristic: aromatic atoms in largest fused system
- Mask/weight non-pocket atoms lower
- Train model to focus on mechanistically-relevant substructures

**Hypothesis:** Activity cliffs are caused by small structural changes in the binding pocket region (Helix-12 flip area). Standard GNNs treat all atoms equally, so they struggle. Mechanistic masking should help.

---

## Architecture
```
Input: Analog pairs (mol_i, mol_j) with ΔpEC50
        ↓
   D-MPNN encoder (shared weights)
        ↓
   [embedding_i, embedding_j]
        ↓
   Contrastive Loss:
   - If ΔpEC50 small: pull together
   - If ΔpEC50 large (activity cliff): push apart
        ↓
   Fine-tuned encoder → downstream regression
```

---

**Day 11 Baseline:** 0.5555 RAE (Chemprop + LGBM ensemble)

**Target:** <0.55 RAE by learning activity cliff features

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict

# RDKit
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# Chemprop
from chemprop import data as cpdata, models as cpmodels, featurizers
import chemprop.nn as cpnn
import lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Sklearn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style('whitegrid')

print("✅ Imports complete")

## 1. Load Data

In [ ]:
# Load training data
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_counter-assay_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\npEC50 distribution:")
print(train_df['pEC50'].describe())

## 2. Find Analog Pairs (Activity Cliffs)

**Definition:** Analog pair = molecules with high structural similarity but large activity difference

**Criteria:**
- Tanimoto similarity ≥ 0.6 (structural analogs)
- ΔpEC50 ≥ 1.0 (significant activity difference, ~10x potency change)

In [ ]:
def calculate_rae(y_true, y_pred):
    """Calculate Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

def get_morgan_fp(smiles, radius=2, nbits=2048):
    """Generate Morgan fingerprint."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)

def tanimoto_similarity(fp1, fp2):
    """Calculate Tanimoto similarity between two fingerprints."""
    if fp1 is None or fp2 is None:
        return 0.0
    return DataStructs.TanimotoSimilarity(fp1, fp2)

print("Generating fingerprints...")
train_df['morgan_fp'] = [get_morgan_fp(s) for s in tqdm(train_df.SMILES)]

# Filter valid molecules
valid_mask = train_df['morgan_fp'].notna()
train_df = train_df[valid_mask].reset_index(drop=True)
print(f"Valid molecules: {len(train_df)}")

In [ ]:
# Find analog pairs with activity cliffs
print("\nFinding analog pairs (this may take a few minutes)...")

# Parameters
MIN_TANIMOTO = 0.6  # Structural similarity threshold
MIN_DELTA_PEC50 = 1.0  # Activity cliff threshold (10x potency difference)

analog_pairs = []
all_pairs = []

# Only compare against a random subset to speed up (or all if dataset is small)
n_comparisons = min(len(train_df), 1500)
comparison_indices = np.random.choice(len(train_df), n_comparisons, replace=False)

for i in tqdm(range(len(train_df))):
    fp_i = train_df.iloc[i]['morgan_fp']
    pec50_i = train_df.iloc[i]['pEC50']
    
    for j in comparison_indices:
        if j <= i:  # Avoid duplicates and self-comparison
            continue
        
        fp_j = train_df.iloc[j]['morgan_fp']
        pec50_j = train_df.iloc[j]['pEC50']
        
        tanimoto = tanimoto_similarity(fp_i, fp_j)
        delta_pec50 = abs(pec50_i - pec50_j)
        
        # Store all pairs for analysis
        all_pairs.append({
            'idx_i': i,
            'idx_j': j,
            'tanimoto': tanimoto,
            'delta_pec50': delta_pec50,
            'pec50_i': pec50_i,
            'pec50_j': pec50_j
        })
        
        # Activity cliff: high similarity, large activity difference
        if tanimoto >= MIN_TANIMOTO and delta_pec50 >= MIN_DELTA_PEC50:
            analog_pairs.append({
                'idx_i': i,
                'idx_j': j,
                'smiles_i': train_df.iloc[i]['SMILES'],
                'smiles_j': train_df.iloc[j]['SMILES'],
                'pec50_i': pec50_i,
                'pec50_j': pec50_j,
                'delta_pec50': delta_pec50,
                'tanimoto': tanimoto
            })

analog_df = pd.DataFrame(analog_pairs)
all_pairs_df = pd.DataFrame(all_pairs)

print(f"\n✅ Found {len(analog_df)} activity cliff pairs")
print(f"   (Tanimoto ≥ {MIN_TANIMOTO}, ΔpEC50 ≥ {MIN_DELTA_PEC50})")
print(f"\nTop 5 activity cliffs:")
print(analog_df.nlargest(5, 'delta_pec50')[['tanimoto', 'delta_pec50', 'pec50_i', 'pec50_j']])

In [ ]:
# Visualize analog pairs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Tanimoto vs ΔpEC50
axes[0].hexbin(all_pairs_df['tanimoto'], all_pairs_df['delta_pec50'], 
               gridsize=50, cmap='Blues', alpha=0.6)
axes[0].scatter(analog_df['tanimoto'], analog_df['delta_pec50'], 
                c='red', s=30, alpha=0.7, label=f'Activity cliffs (n={len(analog_df)})')
axes[0].axhline(MIN_DELTA_PEC50, color='red', linestyle='--', alpha=0.5)
axes[0].axvline(MIN_TANIMOTO, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Tanimoto Similarity', fontsize=12)
axes[0].set_ylabel('ΔpEC50 (Activity Difference)', fontsize=12)
axes[0].set_title('Activity Cliff Landscape', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribution of activity cliffs
axes[1].hist(analog_df['delta_pec50'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('ΔpEC50', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Activity Cliff Magnitude Distribution', fontsize=14, fontweight='bold')
axes[1].axvline(MIN_DELTA_PEC50, color='red', linestyle='--', linewidth=2, label=f'Threshold: {MIN_DELTA_PEC50}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Pocket-Masking: Identify Hot Spot Atoms

**Goal:** Identify which atoms are likely to interact with F288/W299/Y306 hydrophobic π-trap

**Heuristic (without docking):**
1. Find largest aromatic system (most likely to π-stack with F288/W299/Y306)
2. Identify hydrophobic regions near aromatic core
3. These atoms get **higher attention weights** in the contrastive loss

In [ ]:
def identify_pocket_atoms(smiles):
    """
    Heuristic to identify atoms likely to bind in PXR hydrophobic pocket.
    
    Returns:
        pocket_mask: Binary mask (1 = pocket atom, 0 = solvent-exposed)
        pocket_weights: Float weights for each atom (higher = more important)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    
    n_atoms = mol.GetNumAtoms()
    pocket_mask = np.zeros(n_atoms, dtype=int)
    pocket_weights = np.ones(n_atoms, dtype=float)
    
    # Step 1: Find largest aromatic system (π-stacking with F288/W299/Y306)
    aromatic_systems = []
    for ring in mol.GetRingInfo().AtomRings():
        if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            aromatic_systems.append(set(ring))
    
    if aromatic_systems:
        # Find largest fused aromatic system
        largest_aromatic = max(aromatic_systems, key=len)
        
        for atom_idx in largest_aromatic:
            pocket_mask[atom_idx] = 1
            pocket_weights[atom_idx] = 3.0  # High weight for aromatic core
    
    # Step 2: Add hydrophobic atoms near aromatic core
    for atom_idx in range(n_atoms):
        atom = mol.GetAtomWithIdx(atom_idx)
        
        # Hydrophobic atoms (C, aromatic atoms)
        if atom.GetSymbol() in ['C'] and atom.GetIsAromatic():
            pocket_mask[atom_idx] = 1
            pocket_weights[atom_idx] = 2.5
        
        # Check if atom is connected to aromatic core
        for neighbor in atom.GetNeighbors():
            if neighbor.GetIdx() in (largest_aromatic if aromatic_systems else []):
                if atom.GetSymbol() in ['C', 'F', 'Cl', 'Br', 'I']:  # Hydrophobic substituents
                    pocket_mask[atom_idx] = 1
                    pocket_weights[atom_idx] = 2.0
    
    # Step 3: Polar atoms get low weights (likely solvent-exposed)
    for atom_idx in range(n_atoms):
        atom = mol.GetAtomWithIdx(atom_idx)
        if atom.GetSymbol() in ['O', 'N', 'S'] and pocket_mask[atom_idx] == 0:
            pocket_weights[atom_idx] = 0.5  # Low weight for polar, non-pocket atoms
    
    return pocket_mask, pocket_weights

# Test on a sample molecule
test_smiles = train_df.iloc[0]['SMILES']
test_mask, test_weights = identify_pocket_atoms(test_smiles)

print(f"Test molecule: {test_smiles}")
print(f"Number of atoms: {len(test_mask)}")
print(f"Pocket atoms: {test_mask.sum()} ({100*test_mask.sum()/len(test_mask):.1f}%)")
print(f"Weight distribution: min={test_weights.min():.1f}, max={test_weights.max():.1f}, mean={test_weights.mean():.2f}")

In [ ]:
# Add pocket masks to training data
print("Computing pocket masks for all molecules...")

train_df['pocket_mask'] = None
train_df['pocket_weights'] = None

for idx in tqdm(range(len(train_df))):
    smiles = train_df.iloc[idx]['SMILES']
    mask, weights = identify_pocket_atoms(smiles)
    train_df.at[idx, 'pocket_mask'] = mask
    train_df.at[idx, 'pocket_weights'] = weights

print("✅ Pocket masks computed")

# Analyze pocket coverage
pocket_fractions = []
for idx in range(len(train_df)):
    mask = train_df.iloc[idx]['pocket_mask']
    if mask is not None:
        pocket_fractions.append(mask.sum() / len(mask))

print(f"\nPocket atom fraction: {np.mean(pocket_fractions):.2%} ± {np.std(pocket_fractions):.2%}")

## 4. Contrastive Learning Dataset

**Dataset Structure:**
- Positive pairs: Similar molecules with similar activities (ΔpEC50 < 0.5)
- Negative pairs: Similar molecules with different activities (ΔpEC50 ≥ 1.0, activity cliffs)
- Hard negatives: Very similar molecules (Tanimoto > 0.7) with large ΔpEC50 (these are the learning signal!)

In [ ]:
class ContrastivePairDataset(Dataset):
    """
    Dataset of molecular pairs for contrastive learning.
    
    Each sample: (mol_i, mol_j, similarity_label, delta_pec50)
    - similarity_label: 1 if similar activity, 0 if activity cliff
    - delta_pec50: absolute difference in pEC50
    """
    
    def __init__(self, train_df, analog_df, use_pocket_masking=False):
        self.train_df = train_df
        self.analog_df = analog_df
        self.use_pocket_masking = use_pocket_masking
        
        # Create positive and negative pairs
        self.pairs = []
        
        # Negative pairs (activity cliffs)
        for _, row in analog_df.iterrows():
            self.pairs.append({
                'idx_i': row['idx_i'],
                'idx_j': row['idx_j'],
                'label': 0,  # Activity cliff
                'delta_pec50': row['delta_pec50'],
                'tanimoto': row['tanimoto']
            })
        
        # Positive pairs (similar molecules with similar activities)
        # Sample from high Tanimoto, low ΔpEC50 pairs
        positive_candidates = all_pairs_df[
            (all_pairs_df['tanimoto'] >= 0.6) & 
            (all_pairs_df['delta_pec50'] < 0.5)
        ]
        
        # Balance positive/negative pairs
        n_positive = min(len(positive_candidates), len(self.pairs))
        positive_sample = positive_candidates.sample(n=n_positive, random_state=42)
        
        for _, row in positive_sample.iterrows():
            self.pairs.append({
                'idx_i': row['idx_i'],
                'idx_j': row['idx_j'],
                'label': 1,  # Similar activity
                'delta_pec50': row['delta_pec50'],
                'tanimoto': row['tanimoto']
            })
        
        print(f"Created dataset with {len(self.pairs)} pairs:")
        print(f"  - Negative (activity cliffs): {sum(p['label'] == 0 for p in self.pairs)}")
        print(f"  - Positive (similar activity): {sum(p['label'] == 1 for p in self.pairs)}")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Get molecules
        mol_i = self.train_df.iloc[pair['idx_i']]
        mol_j = self.train_df.iloc[pair['idx_j']]
        
        data = {
            'smiles_i': mol_i['SMILES'],
            'smiles_j': mol_j['SMILES'],
            'pec50_i': mol_i['pEC50'],
            'pec50_j': mol_j['pEC50'],
            'label': pair['label'],
            'delta_pec50': pair['delta_pec50'],
            'tanimoto': pair['tanimoto']
        }
        
        if self.use_pocket_masking:
            data['pocket_weights_i'] = mol_i['pocket_weights']
            data['pocket_weights_j'] = mol_j['pocket_weights']
        
        return data

# Create dataset
contrastive_dataset = ContrastivePairDataset(
    train_df=train_df,
    analog_df=analog_df,
    use_pocket_masking=True
)

print(f"\n✅ Contrastive dataset created with {len(contrastive_dataset)} pairs")

## 5. Contrastive Loss Function

**Custom Loss:** Weighted contrastive loss with activity-aware margin

```python
If activity cliff (ΔpEC50 large):
    Push embeddings apart: loss = max(0, margin - ||emb_i - emb_j||)
    
If similar activity (ΔpEC50 small):
    Pull embeddings together: loss = ||emb_i - emb_j||²
    
Margin = f(ΔpEC50)  # Adaptive margin based on activity difference
```

For pocket-masking: weight loss by average pocket importance

In [ ]:
class ActivityAwareContrastiveLoss(nn.Module):
    """
    Contrastive loss that adapts margin based on activity difference.
    
    - Small ΔpEC50: pull embeddings together
    - Large ΔpEC50 (activity cliff): push embeddings apart with adaptive margin
    """
    
    def __init__(self, base_margin=1.0, margin_scale=0.5):
        super().__init__()
        self.base_margin = base_margin
        self.margin_scale = margin_scale
    
    def forward(self, emb_i, emb_j, label, delta_pec50):
        """
        Args:
            emb_i, emb_j: Embeddings (batch_size, embedding_dim)
            label: 1 if similar activity, 0 if activity cliff (batch_size,)
            delta_pec50: Activity difference (batch_size,)
        """
        # Euclidean distance between embeddings
        distances = F.pairwise_distance(emb_i, emb_j)
        
        # Adaptive margin: larger activity difference → larger margin
        margin = self.base_margin + self.margin_scale * delta_pec50
        
        # Contrastive loss
        # Similar activity (label=1): minimize distance
        positive_loss = label * torch.pow(distances, 2)
        
        # Activity cliff (label=0): maximize distance up to margin
        negative_loss = (1 - label) * torch.pow(torch.clamp(margin - distances, min=0.0), 2)
        
        loss = positive_loss + negative_loss
        return loss.mean()

# Test loss function
test_loss_fn = ActivityAwareContrastiveLoss()
test_emb_i = torch.randn(4, 128)
test_emb_j = torch.randn(4, 128)
test_labels = torch.tensor([1, 1, 0, 0], dtype=torch.float32)
test_deltas = torch.tensor([0.3, 0.4, 1.5, 2.0], dtype=torch.float32)

test_loss = test_loss_fn(test_emb_i, test_emb_j, test_labels, test_deltas)
print(f"✅ Test loss: {test_loss.item():.4f}")

## 6. Contrastive Pretraining Model

**Architecture:**
1. D-MPNN encoder (from Chemprop)
2. Projection head (maps to embedding space for contrastive learning)
3. After pretraining: remove projection head, finetune encoder for regression

In [ ]:
class ContrastiveMPNN(pl.LightningModule):
    """
    D-MPNN with contrastive learning objective.
    
    Training: Learn embeddings that separate activity cliffs
    Inference: Use learned encoder for downstream regression
    """
    
    def __init__(self, hidden_dim=300, embedding_dim=128, learning_rate=1e-4, 
                 use_pocket_masking=False):
        super().__init__()
        self.save_hyperparameters()
        
        # D-MPNN encoder
        self.message_passing = cpnn.BondMessagePassing(d_h=hidden_dim)
        self.aggregation = cpnn.MeanAggregation()
        
        # Projection head for contrastive learning
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )
        
        # Contrastive loss
        self.contrastive_loss = ActivityAwareContrastiveLoss()
        
        self.learning_rate = learning_rate
        self.use_pocket_masking = use_pocket_masking
    
    def encode(self, batch):
        """Encode molecules to embeddings."""
        # Message passing
        h = self.message_passing(batch)
        
        # Aggregation
        mol_emb = self.aggregation(h, batch.batch)
        
        # Project to contrastive space
        emb = self.projection(mol_emb)
        
        return emb
    
    def forward(self, batch_i, batch_j):
        """Forward pass for molecule pairs."""
        emb_i = self.encode(batch_i)
        emb_j = self.encode(batch_j)
        return emb_i, emb_j
    
    def training_step(self, batch, batch_idx):
        # Unpack batch (this is a simplified version - actual implementation needs proper batching)
        batch_i, batch_j, labels, delta_pec50 = batch
        
        # Get embeddings
        emb_i, emb_j = self.forward(batch_i, batch_j)
        
        # Compute loss
        loss = self.contrastive_loss(emb_i, emb_j, labels, delta_pec50)
        
        self.log('train_loss', loss)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

print("✅ Contrastive MPNN model defined")

## 7. Training Strategy

**Two-stage training:**

### Stage 1: Contrastive Pretraining (10-20 epochs)
- Train on analog pairs with contrastive loss
- Learn embeddings that separate activity cliffs
- Use pocket-masking to focus on mechanistically-relevant atoms

### Stage 2: Supervised Finetuning (30-50 epochs)
- Remove projection head
- Add regression head (predict pEC50)
- Finetune encoder + regression head on full training set

**Hypothesis:** Contrastive pretraining will help model learn subtle structural differences that drive activity cliffs, improving blind test performance.

---

## Implementation Notes

This notebook provides the **framework** for contrastive learning on activity cliffs. Full implementation requires:

1. **Custom DataLoader** for molecule pairs with Chemprop's graph featurization
2. **Proper batching** of molecular graphs (use `torch_geometric` batching)
3. **Pocket-masking integration** into message passing (weight atom embeddings)
4. **Two-stage training loop** with checkpoint management
5. **Evaluation** on downstream regression task

**Next steps:**
- Implement full training pipeline
- Compare pocket-masked vs full-molecule contrastive learning
- Ensemble contrastive-finetuned model with Day 11 baseline
- Analyze which activity cliffs the model learns to resolve

In [ ]:
# Placeholder for training loop (full implementation needed)
print("="*80)
print("TRAINING PIPELINE OUTLINE")
print("="*80)

print("""
Stage 1: Contrastive Pretraining
--------------------------------
1. Create DataLoader from ContrastivePairDataset
2. Initialize ContrastiveMPNN model
3. Train with activity-aware contrastive loss
4. Monitor: embedding distance for positive/negative pairs
5. Save pretrained encoder weights

Stage 2: Supervised Finetuning  
------------------------------
1. Load pretrained encoder
2. Remove projection head, add RegressionFFN
3. Finetune on full training set (pEC50 regression)
4. Use scaffold-based CV for validation
5. Compare RAE with baseline (Day 11: 0.5555)

Ablation Studies:
----------------
A. Full-molecule contrastive (no pocket masking)
B. Pocket-masked contrastive (weight aromatic core 3x)
C. No contrastive pretraining (baseline)
D. Different Tanimoto/ΔpEC50 thresholds for activity cliffs

Expected Improvements:
---------------------
- Better handling of activity cliffs in blind test
- Lower RAE on structurally similar test molecules
- More interpretable embeddings (activity cliffs separated)
""")

print("="*80)
print("To implement full pipeline, see Chemprop docs:")
print("https://chemprop.readthedocs.io/en/latest/")
print("="*80)

## 8. Alternative: SimCLR-style Augmentation

**Another approach:** Instead of analog pairs, use **SMILES augmentation** + contrastive learning

```python
# Generate multiple SMILES for same molecule (via RDKit canonicalization randomization)
mol = Chem.MolFromSmiles(smiles)
aug_smiles_1 = Chem.MolToSmiles(mol, doRandom=True)
aug_smiles_2 = Chem.MolToSmiles(mol, doRandom=True)

# Contrastive loss: augmented views should have similar embeddings
# This forces model to learn canonical molecular representations
```

Combined with activity cliff pairs, this could improve generalization.

## Summary

**Key Ideas:**
1. ✅ Activity cliffs are hard for standard models to learn
2. ✅ Contrastive learning on analog pairs teaches model to distinguish subtle differences
3. ✅ Pocket-masking focuses learning on mechanistically-relevant atoms (F288/W299/Y306 binding)
4. ✅ Two-stage training: contrastive pretraining → supervised finetuning

**Framework provided:**
- ✅ Analog pair detection (Tanimoto + ΔpEC50)
- ✅ Pocket atom identification heuristic
- ✅ Activity-aware contrastive loss
- ✅ Contrastive MPNN architecture

**Next steps:**
- Implement full training pipeline with proper Chemprop integration
- Run ablation: pocket-masked vs full-molecule contrastive
- Ensemble with Day 11 baseline (Chemprop + LGBM)
- Target: <0.55 RAE on blind test

**Key insight:** Activity cliffs happen in the **binding pocket**, not the solvent-exposed tail. Pocket-masking should help the model learn what matters for PXR agonism (Helix-12 flip mechanism).